# Vision Transformers (ViT) — практический пример в Google Colab

**Цель:** запустить готовый Vision Transformer, увидеть патчи и эмбеддинги, получить Top-5 классов и посмотреть на веса self-attention.

**Что понадобится:** Google Colab, интернет для первой загрузки весов (~23 МБ) и изображения. GPU **необязателен**: пример использует компактную модель ViT-Tiny.

**Модель:** [`timm/vit_tiny_patch16_224.augreg_in21k_ft_in1k`](https://huggingface.co/timm/vit_tiny_patch16_224.augreg_in21k_ft_in1k) — изображение 224×224, патчи 16×16, 1000 классов ImageNet. Модель уже обучена; мы делаем только inference.

## 1. Установка и импорт библиотек

In [ ]:
import io
from urllib.request import urlopen

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch
import torch.nn.functional as F
import timm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
print("PyTorch:", torch.__version__)
print("timm:", timm.__version__)

## 2. Загружаем pretrained ViT

**ViT-Tiny** — облегчённый вариант классического Vision Transformer. Размеры ниже относятся именно к этой модели, а не ко всем ViT.

Для первого запуска веса скачиваются из интернета.

In [ ]:
MODEL_NAME = "vit_tiny_patch16_224.augreg_in21k_ft_in1k"
model = timm.create_model(MODEL_NAME, pretrained=True).to(DEVICE).eval()

config = timm.data.resolve_model_data_config(model)
transform = timm.data.create_transform(**config, is_training=False)

print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f} M")
print("Image size:", config["input_size"])
print("Patch size:", model.patch_embed.patch_size)
print("Embedding size:", model.num_features)
print("Transformer blocks:", len(model.blocks))

## 3. Загружаем тестовую картинку

По умолчанию используем общедоступный пример с попугаями. Можно заменить `IMAGE_URL` на ссылку на другое изображение или запустить необязательную ячейку загрузки собственного файла ниже.

In [ ]:
IMAGE_URL = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/hub/parrots.png"
image = Image.open(io.BytesIO(urlopen(IMAGE_URL).read())).convert("RGB")

plt.figure(figsize=(7, 5))
plt.imshow(image)
plt.title("Original image")
plt.axis("off")
plt.show()

In [ ]:
from pathlib import Path
from PIL import Image
from IPython.display import display

# Путь к локальному изображению
image_path = Path("/home/mu/WORK/Univer/CV/test1.jpg")

# Загрузка изображения
image = Image.open(image_path).convert("RGB")

display(image)

## 4. Разбиение изображения на патчи

Для 224×224 и patch size 16×16 получается **14×14 = 196 патчей**. Это не 196 отдельных входных изображений в модели: `Patch Embedding` переводит каждый патч в вектор признаков.

Изображение ниже приведено к размеру входа модели **для визуализации сетки**. Фактическая предобработка модели может включать resize и crop.

In [ ]:
PATCH = model.patch_embed.patch_size[0]
SIZE = config["input_size"][-1]
visual_image = image.resize((SIZE, SIZE))

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(visual_image)
for p in range(0, SIZE + 1, PATCH):
    ax.axhline(p - 0.5, color="white", alpha=0.7, linewidth=0.7)
    ax.axvline(p - 0.5, color="white", alpha=0.7, linewidth=0.7)
ax.set_title(f"{SIZE}×{SIZE} → {SIZE // PATCH}×{SIZE // PATCH} = {(SIZE // PATCH) ** 2} patches")
ax.axis("off")
plt.show()

## 5. Patch Embedding и `[CLS]` токен

После предобработки `Patch Embedding` преобразует каждый патч в вектор размерности **192**. К последовательности добавляется обучаемый `[CLS]`-токен для классификации. Поэтому в трансформере участвуют **197 токенов**: 196 патчей + 1 `[CLS]`.

Также к токенам прибавляются позиционные эмбеддинги, чтобы модель учитывала расположение патчей.

In [ ]:
x = transform(image).unsqueeze(0).to(DEVICE)
with torch.inference_mode():
    patch_tokens = model.patch_embed(x)
    encoded_tokens = model.forward_features(x)

print("Input tensor:        ", tuple(x.shape))
print("Patch embeddings:    ", tuple(patch_tokens.shape))
print("ViT output tokens:   ", tuple(encoded_tokens.shape))
print("[CLS] token features:", tuple(encoded_tokens[:, 0].shape))
assert patch_tokens.shape[1] == 196 and encoded_tokens.shape[1] == 197

## 6. Классификация: Top-5 предсказаний

Модель возвращает 1000 чисел (`logits`) — по одному для каждого класса ImageNet. `softmax` переводит их в значения от 0 до 1. Это **оценки уверенности модели**, а не гарантия правильности ответа.

In [ ]:
LABELS_URL = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
labels = urlopen(LABELS_URL).read().decode("utf-8").splitlines()
assert len(labels) == 1000

with torch.inference_mode():
    logits = model(x)
    probs = torch.softmax(logits, dim=1)[0]

values, indices = probs.topk(5)
for rank, (score, index) in enumerate(zip(values.cpu(), indices.cpu()), start=1):
    print(f"{rank}. {labels[index.item()]:30s}  {score.item():.1%}")

plt.figure(figsize=(8, 3))
plt.barh([labels[i.item()] for i in indices.cpu()][::-1], values.cpu().numpy()[::-1])
plt.xlabel("Model confidence")
plt.title("Top-5 ImageNet predictions")
plt.tight_layout()
plt.show()

## 7. Что показывает self-attention?

Веса self-attention описывают, **на какие токены смотрит данный токен при обновлении своего представления**. В ViT это матрица для каждой attention head в каждом слое.

Для визуализации возьмём **последний Transformer-блок** и внимание токена `[CLS]` к патчам. Покажем одну head и среднее по всем heads.

> **Важно:** attention-карта — это не доказательство, что именно эти области стали причиной предсказания. Она показывает внутренние веса одного механизма модели, но не является надёжной объяснимостью сама по себе.

In [ ]:
# В timm при ускоренном fused attention матрица attention не возвращается.
# Переключаем ПОСЛЕДНИЙ блок в обычный режим и читаем выход attention-dropout.
last_attn = model.blocks[-1].attn
last_attn.fused_attn = False
captured = {}

def capture_attention(module, inputs, output):
    captured["weights"] = output.detach().cpu()

handle = last_attn.attn_drop.register_forward_hook(capture_attention)
try:
    with torch.inference_mode():
        _ = model(x)
finally:
    handle.remove()

if "weights" not in captured:
    raise RuntimeError("Не удалось извлечь attention: проверьте версию timm.")

attn = captured["weights"]          # [batch, heads, tokens, tokens]
print("Attention matrix shape:", tuple(attn.shape))
print("Interpretation: [batch, heads, query tokens, key tokens]")

n_patches_side = SIZE // PATCH
assert attn.shape[-1] == n_patches_side ** 2 + 1

# Строка 0 — внимание от [CLS] ко всем входным токенам.
# Колонку 0 ([CLS] -> [CLS]) исключаем из карты патчей.
cls_to_patches = attn[0, :, 0, 1:]
head_map = cls_to_patches[0].reshape(n_patches_side, n_patches_side).numpy()
mean_map = cls_to_patches.mean(dim=0).reshape(n_patches_side, n_patches_side).numpy()

# Для overlay используем то же подготовленное изображение, что и для модели.
# Преобразуем нормализованный тензор обратно в RGB (для визуализации).
mean = torch.tensor(config["mean"]).view(3, 1, 1)
std = torch.tensor(config["std"]).view(3, 1, 1)
vis_tensor = (x[0].detach().cpu() * std + mean).clamp(0, 1)
vis_img = vis_tensor.permute(1, 2, 0).numpy()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(vis_img)
axes[0].set_title("Input image (model crop)")
axes[1].imshow(vis_img)
axes[1].imshow(head_map, cmap="jet", alpha=0.55, extent=(0, SIZE, SIZE, 0), interpolation="bilinear")
axes[1].set_title("Last block: attention head 0")
axes[2].imshow(vis_img)
axes[2].imshow(mean_map, cmap="jet", alpha=0.55, extent=(0, SIZE, SIZE, 0), interpolation="bilinear")
axes[2].set_title("Last block: mean of all heads")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 8. Эксперименты

1. Замените изображение и сравните Top-5 предсказаний.
2. Попробуйте посмотреть **разные attention heads** — поменяйте `cls_to_patches[0]` на `[1]`, `[2]` и т. д. Почему карты различаются?
3. Сделайте изображение с объектом на сложном фоне. Изменятся ли attention-карты и вероятность правильного класса?
4. **Вопрос для обсуждения:** можно ли считать attention-карту полноценным объяснением решения модели? Почему нет?

### Ключевые выводы
- ViT работает с последовательностью векторов-патчей, а не с отдельными словами.
- Self-attention позволяет учитывать связи между разными частями изображения.
- `[CLS]` агрегирует информацию для классификации, но не является единственным способом построения ViT.
- Pretrained-модель удобно использовать для inference; обучение с нуля потребует значительно больше данных и ресурсов.

### Источники
- [Оригинальная статья ViT (2020)](https://arxiv.org/abs/2010.11929)
- [Статья Habr для лекции](https://habr.com/ru/articles/922868/)
- [Карточка использованной модели](https://huggingface.co/timm/vit_tiny_patch16_224.augreg_in21k_ft_in1k)
- [Библиотека timm](https://github.com/huggingface/pytorch-image-models)
